# Session 08 · Replication and Sharding

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - majority and elections

In [2]:
import random
from cluster_sim import ReplicaSet, hashed_router, ranged_router, route_counts, score_key, shards_for_query
rs = ReplicaSet(["db1", "db2", "db3"])
print("majority needed:", rs.majority(), "| primary:", rs.primary)
rs.fail("db1")
print("db1 down -> new primary:", rs.elect())
rs.fail("db2")
print("db2 down too -> primary:", rs.elect(), "| can write?", rs.can_write())

majority needed: 2 | primary: db1
db1 down -> new primary: db2
db2 down too -> primary: None | can write? False


## Example 2 - write concern

In [3]:
rs = ReplicaSet(["db1", "db2", "db3"])
print(rs.write({"x": 1}, w=1), rs.write({"x": 2}, w="majority"))
rs.fail("db3")
print("one secondary down, majority still ok:", rs.write({"x": 3}, w="majority"))
try:
    rs.write({"x": 4}, w=3)
except TimeoutError as e:
    print("w=3:", e)

{'ok': 1, 'acknowledged_by': 1} {'ok': 1, 'acknowledged_by': 2}
one secondary down, majority still ok: {'ok': 1, 'acknowledged_by': 2}
w=3: write concern 3 not satisfied (2 healthy)


## Example 3 - ranged sharding with a monotonic key

In [4]:
existing_max = 900_000
new_ids = range(existing_max, existing_max + 30_000)
print("ranged:", route_counts(new_ids, ranged_router([300_000, 600_000])))

ranged: {2: 30000}


## Example 4 - hashed sharding spreads writes

In [5]:
print("hashed:", route_counts(new_ids, hashed_router(3)))

hashed: {0: 10057, 1: 9994, 2: 9949}


## Example 5 - targeted vs scatter-gather

In [6]:
print(shards_for_query({"customer_id": 42}, key="customer_id"))
print(shards_for_query({"city": "Karachi"}, key="customer_id"))

[1]
[0, 1, 2]


## Example 6 - scoring shard-key candidates

In [7]:
rng = random.Random(8)
orders = [{"city": rng.choices(["Karachi", "Lahore", "Islamabad", "Peshawar", "Quetta"], [60, 20, 10, 6, 4])[0],
           "customer_id": rng.randrange(500_000)} for _ in range(20_000)]
print(score_key(orders, "city"), score_key(orders, "customer_id"), sep="\n")

{'field': 'city', 'cardinality': 5, 'max_share': 0.603}
{'field': 'customer_id', 'cardinality': 19584, 'max_share': 0.0}
